# Example-15: Set TEST PVs

In [1]:
# Import

import epics
import numpy
import torch

import sys
sys.path.append('..')

from harmonica.util import pv_make
from harmonica.window import Window
from harmonica.data import Data

import matplotlib.pyplot as plt
from time import sleep

torch.set_printoptions(precision=12, sci_mode=True)
print(torch.cuda.is_available())
print(torch.get_num_threads())

True
16


In [2]:
# Set data type and device

dtype = torch.float64
device = torch.device('cpu')

In [3]:
# Start TEST PVs (softIoc -d harmonica.db)
# Load test TbT data, add orbit/noise/spikes and save to TEST PVs

import epics
import numpy
import torch
import pandas

import sys
sys.path.append('..')

from harmonica.util import pv_make
from harmonica.window import Window
from harmonica.data import Data

torch.set_printoptions(precision=12, sci_mode=True)
print(torch.cuda.is_available())

dtype = torch.float64
device = torch.device('cpu')

df = pandas.read_pickle('../virtual_tbt.pkl.gz')

w = Window(4096, dtype=dtype, device=device)
x = Data.from_data(w, torch.tensor(df.X.to_list(), dtype=dtype, device=device))
y = Data.from_data(w, torch.tensor(df.Y.to_list(), dtype=dtype, device=device))

noise_x = 1.0E-6*(25.0 + 50.0*torch.rand(x.size, dtype=dtype, device=device))
noise_y = 1.0E-6*(25.0 + 50.0*torch.rand(y.size, dtype=dtype, device=device))

bpm = ['STP2', 'STP4', 'SRP1', 'SRP2', 'SRP3', 'SRP4', 'SRP5', 'SRP6', 'SRP7', 'SRP8', 'SRP9', 'SIP1', 'SIP2', 'SRP10', 'SRP11', 'SRP12', 'SRP13', 'SRP14', 'SRP15', 'SRP16', 'SRP17', 'SEP5', 'SEP4', 'SEP3', 'SEP1', 'SEP0', 'NEP0', 'NEP1', 'NEP3', 'NEP4', 'NEP5', 'NRP17', 'NRP16', 'NRP15', 'NRP14', 'NRP13', 'NRP12', 'NRP11', 'NRP10', 'NIP3', 'NIP1', 'NRP9', 'NRP8', 'NRP7', 'NRP6', 'NRP5', 'NRP4', 'NRP3', 'NRP2', 'NRP1', 'NTP4', 'NTP2', 'NTP0', 'STP0']

length = 2048
w = Window(length, 'cosine_window', 1.0, dtype=dtype, device=device)
d = Data.from_data(w, x.data[:, :length])
d.add_noise(noise_x)
d.data.copy_(d.work)
pv_list = [pv_make(name, 'X', True) for name in bpm]
pv_rise = [0 for _ in range(len(bpm))]
d.source = 'epics'
d.pv_list = pv_list
d.pv_rise = pv_rise
d.save_epics()

length = 2048
w = Window(length, 'cosine_window', 1.0, dtype=dtype, device=device)
d = Data.from_data(w, y.data[:, :length])
d.add_noise(noise_y)
d.data.copy_(d.work)
pv_list = [pv_make(name, 'Y', True) for name in bpm]
pv_rise = [0 for _ in range(len(bpm))]
d.source = 'epics'
d.pv_list = pv_list
d.pv_rise = pv_rise
d.save_epics()

True


In [4]:
# Start TEST PVs (softIoc -d harmonica.db)
# Load test TbT data, add orbit/noise/spikes and save to TEST PVs
# And start errors

import epics
import numpy
import torch
import pandas

import sys
sys.path.append('..')

from harmonica.util import pv_make
from harmonica.window import Window
from harmonica.data import Data

torch.set_printoptions(precision=12, sci_mode=True)
print(torch.cuda.is_available())

dtype = torch.float64
device = torch.device('cpu')

df = pandas.read_pickle('../virtual_tbt.pkl.gz')

w = Window(4096, dtype=dtype, device=device)
x = Data.from_data(w, torch.tensor(df.X.to_list(), dtype=dtype, device=device))
y = Data.from_data(w, torch.tensor(df.Y.to_list(), dtype=dtype, device=device))

start = 128
error = {10: 127, 31: 129, 32:129, 41:127, 43:129}
error = {}
count = 2048
data_x = torch.zeros((54, count), dtype=dtype, device=device)
data_y = torch.zeros((54, count), dtype=dtype, device=device)
for i in range(54):
    first = start if i not in error else error[i]
    data_x[i] = x.work[i, first: first + count]
    data_y[i] = y.work[i, first: first + count]

noise_x = 1.0E-6*(25.0 + 50.0*torch.rand(x.size, dtype=dtype, device=device))
noise_y = 1.0E-6*(25.0 + 50.0*torch.rand(y.size, dtype=dtype, device=device))

bpm = ['STP2', 'STP4', 'SRP1', 'SRP2', 'SRP3', 'SRP4', 'SRP5', 'SRP6', 'SRP7', 'SRP8', 'SRP9', 'SIP1', 'SIP2', 'SRP10', 'SRP11', 'SRP12', 'SRP13', 'SRP14', 'SRP15', 'SRP16', 'SRP17', 'SEP5', 'SEP4', 'SEP3', 'SEP1', 'SEP0', 'NEP0', 'NEP1', 'NEP3', 'NEP4', 'NEP5', 'NRP17', 'NRP16', 'NRP15', 'NRP14', 'NRP13', 'NRP12', 'NRP11', 'NRP10', 'NIP3', 'NIP1', 'NRP9', 'NRP8', 'NRP7', 'NRP6', 'NRP5', 'NRP4', 'NRP3', 'NRP2', 'NRP1', 'NTP4', 'NTP2', 'NTP0', 'STP0']

length = 2048
w = Window(length, 'cosine_window', 1.0, dtype=dtype, device=device)
d = Data.from_data(w, data_x[:, :length])
d.add_noise(noise_x)
d.data.copy_(d.work)
pv_list = [pv_make(name, 'X', True) for name in bpm]
pv_rise = [0 for _ in range(len(bpm))]
d.source = 'epics'
d.pv_list = pv_list
d.pv_rise = pv_rise
d.save_epics()

length = 2048
w = Window(length, 'cosine_window', 1.0, dtype=dtype, device=device)
d = Data.from_data(w, data_y[:, :length])
d.add_noise(noise_y)
d.data.copy_(d.work)
pv_list = [pv_make(name, 'Y', True) for name in bpm]
pv_rise = [0 for _ in range(len(bpm))]
d.source = 'epics'
d.pv_list = pv_list
d.pv_rise = pv_rise
d.save_epics()

True
